# Group 10 — Vận hành M1 arXiv-first (A)

**CPU-only · một runtime/writer · source mọi version + revision history · S2 tắt.**
Notebook này dùng Batch/Benchmark hiện có, không dùng corpus Pilot/Comparison.

1. Chọn Runtime → Change runtime type → CPU / accelerator None. Dừng crawler/runtime
   khác dùng cùng corpus. Sau khi bộ bàn giao được review/commit/push, điền full SHA
   của commit đó vào `CODE_COMMIT` ở Setup. Bấm nút Run bình thường của ô Setup.
2. Ở ô Run M1, chỉ chỉnh **CONFIG**: Drive root mới và cửa sổ cohort. Mặc định một
   cohort 100 ID; kiểm tra kết quả trước khi tăng số cohort/lượt. Không bấm Run all
   khi chưa xem cấu hình. Setup mount/clone/cài dependencies, không crawl.
3. Bấm Run M1. Chưa có batch → crawl; còn thiếu và checkpoint hợp lệ → Resume;
   pha arXiv đã đủ → chỉ Audit, không gọi Batch/API lại. Cùng root/commit/cấu hình
   được giữ khi Resume; không sửa JSON checkpoint bằng tay.

**Phạm vi:** 200 cohort × 100 vị trí, giữ thứ tự/mapping từ config gốc:
1–50: 24120010; 51–100: 24120035; 101–150: 24120232; 151–200: 24120260.
Đây là 20.000 ID ứng viên, không phải cam kết 20.000 paper hợp lệ.
Muốn chạy cohort khác chỉ đổi `first_cohort` / `last_cohort` tại CONFIG (bao gồm hai đầu).

**Dừng:** 403/429, lỗi nghiêm trọng hoặc cohort còn thiếu sẽ dừng cửa sổ. Không tự
lặp lại. Không bấm liên tục sau rate limit; đọc audit/logs, chờ và xem xét trước
lượt Run mới. Checksum/checkpoint/orphan lỗi được giữ nguyên để review.
Sau khi Colab ngắt, chắc chắn runtime cũ đã dừng, chạy lại Setup đúng commit rồi
Run M1 cùng root/cohort. Không đồng thời chạy từ hai notebook/tài khoản.

In [ ]:
# SETUP — điền một lần full SHA của commit đã chứa bộ bàn giao này.
CODE_COMMIT = "PASTE_REVIEWED_40_CHARACTER_COMMIT"

import os
from pathlib import Path
import platform
import re
import subprocess
import sys

if re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT) is None:
    raise ValueError("Điền full 40-character commit đã review/push; không dùng commit cũ thiếu bộ bàn giao.")
if os.environ.get("COLAB_GPU") == "1":
    raise RuntimeError("Chuyển accelerator về None/CPU trước khi chạy.")
from google.colab import drive
drive.mount("/content/drive")

REPO = Path("/content/group10-m1-production")
def git_read(*args):
    return subprocess.check_output(["git", "-C", str(REPO), *args], text=True).strip()

if not REPO.exists():
    subprocess.run(["git", "clone", "--no-checkout",
        "https://github.com/Toan-ten/Arxiv-Project-Group10.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_COMMIT], check=True)
if git_read("rev-parse", "HEAD") != CODE_COMMIT or git_read("status", "--porcelain", "--untracked-files=normal"):
    raise RuntimeError("Checkout khác commit hoặc chưa sạch. Không tự reset/pull; xem lại checkout/runtime.")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements-colab.txt")], check=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name, module in tuple(sys.modules.items()):
    if name == "src" or name.startswith("src.") or name == "scripts" or name.startswith("scripts."):
        filename = getattr(module, "__file__", None)
        if filename and not Path(filename).resolve().is_relative_to(REPO):
            raise RuntimeError("Kernel còn module từ checkout khác. Restart runtime rồi chạy lại Setup.")
from scripts.m1_production import ProductionRun
import psutil
print("Setup xong. Python:", platform.python_version(), "| psutil:", psutil.__version__)
print("Commit:", CODE_COMMIT, "| CPU-only: chọn accelerator None trong Runtime.")
print("Chưa crawl. Kiểm tra CONFIG ở ô Run M1; không cần nạp S2 key.")

In [ ]:
# RUN M1 — toàn bộ cấu hình cohort tại đây; mặc định chỉ 100 ID.
CONFIG = {
    "drive_root": "/content/drive/MyDrive/Group10/M1-arxiv-production-v1",
    "first_cohort": 1,
    "last_cohort": 1,
    "timeout": 60.0,
    "sample_interval": 1.0,
    "disk_interval": 120.0,
}

if not Path(CONFIG["drive_root"]).resolve().is_relative_to(Path("/content/drive/MyDrive").resolve()):
    raise ValueError("drive_root phải nằm trên MyDrive, ngoài thư mục code/Pilot/Comparison.")
operator = ProductionRun(REPO, CONFIG["drive_root"], CODE_COMMIT,
    timeout=CONFIG["timeout"], sample_interval=CONFIG["sample_interval"], disk_interval=CONFIG["disk_interval"])
progress = operator.run(CONFIG["first_cohort"], CONFIG["last_cohort"])
print("Bằng chứng bền vững:", CONFIG["drive_root"], "| đọc audits/, logs/, reports/ trước cohort tiếp theo.")

## Đọc kết quả và bàn giao

Mỗi cohort in `mode` (new/resume/audit), `arxiv_phase_complete`, found/not_found,
source success/no_tex/pending/error, HTTP attempts, timing và RAM/disk. Mở
`audit_path` trong Drive để xem đầy đủ; reports và logs giữ từng attempt riêng.
`arxiv_phase_complete=true` chấp nhận not_found được xác nhận, hoặc found với mọi
source/version và history đã verified. Publication/metadata vẫn có thể partial,
references chưa thu thập; **arXiv-first chưa phải hoàn tất M1**.

Tất cả corpus/checkpoint, logs, reports, audits và scratch ở Drive root. Không
đưa vào Git. Root chưa thuộc chiến dịch hoặc cấu hình thay đổi bị từ chối; không
tự sửa file để vượt kiểm tra. Một lần audit chỉ thêm audit JSON mới.

Benchmark: callback wall time gồm pacing/retry/checkpoint, không gồm Setup và audit
ngoài callback. RAM peak/disk peak là đỉnh lấy mẫu; RAM mean là trung bình RSS các
mẫu của kernel + process con. Disk đo toàn corpus tích lũy + scratch, không gồm
logs/reports. Final size là snapshot cuối, không cộng nhiều snapshot. Khi tổng hợp
cohort/attempt: cộng time intervals duy nhất, max peaks cùng scope, RAM mean theo
sample counts; không cộng report lũy kế. Mất số đo do hard kill phải giữ unknown.
Theo dõi quota Drive; notebook không tự bảo đảm dung lượng đủ cho toàn bộ corpus.

Giao C thư mục audits + logs + reports, commit và corpus đã kiểm integrity. S2
enrichment và benchmark cuối gồm S2 là công việc tiếp theo, không chạy ở notebook
này. [Hướng dẫn chi tiết](https://github.com/Toan-ten/Arxiv-Project-Group10/blob/feat/m1-integration/docs/m1_production.md).